# Q1–Q3 Revizyonu: SHAP–LIME açıklama güvenilirliği ve kalibrasyon

Bu *tamamlayıcı* notebook, PAPER deneyinizin **aynı dış katlarını, optimizasyon sonucu seçilen aynı hiperparametreleri ve aynı eğitim/dış test ayırımını** yeniden kurar. Kullanılacak ZIP dosyası: `kredi_ek_deney_sonuclari_paper.zip`.

**Temel bilimsel soru:** Aynı gözlem için aynı modelde kalibrasyon, SHAP–LIME özellik seçimi uyumunu nasıl etkiliyor? Etki sınıf ağırlığına ve kredi veri kümesine göre değişiyor mu?

**Metodolojik iyileştirme:** SHAP ve LIME, *aynı pozitif sınıf olasılığı* çıktısını açıklar. SHAP, modelden bağımsız permutation yöntemiyle **ham orijinal 20/23 özellik** üzerinde hesaplanır; LIME aynı özgün özellikleri kullanır. Dolayısıyla önceki log-odds vs probability ölçek farkı bulunmaz. SHAP ve LIME'ın özelliklere atfettiği işaretlerin semantiği (özellikle LIME binleme/kategori koşullarında) yine tam eşdeğer değildir; bu nedenle yalnızca **mutlak önem sıra/küme uyumu** karşılaştırılır. Deneylerin tümü dış test örneklerinde yürütülür.

**Uyarılar:** Kredi verilerinde tarihsel ve veri kümesi kaynaklı kısıtlar devam eder. Sonuçları yorumlarken veri kümeleri arası nedensel ya da güncel dağılıma genelleme iddiasında bulunmayın. XAI yöntemlerine özgü örnekleme/maskeleme kısıtları ayrı yazılmalıdır.

In [ ]:
%pip -q install "ucimlrepo>=0.0.7" "xgboost>=2.0,<4" "scikit-learn>=1.4,<2" "shap>=0.46,<1" "lime>=0.2.0.1" "requests>=2.28"

## 1. Önce PAPER sonuç ZIP dosyasını yükleyin

Varsa `/content/kredi_ek_deney_sonuclari_paper.zip` okunur, yoksa Colab dosya seçicisi açılır. `MODE='FAST'` ilk kontrol, `MODE='PAPER'` makale için yeterli örneklem sayısıyla çalışır. PAPER yaklaşık yüzlerce yerel açıklama ve LIME tekrarı hesaplar, uzun sürebilir.

In [ ]:
from pathlib import Path
import io, zipfile, json, itertools, time, warnings
import numpy as np, pandas as pd
import requests, scipy, sklearn, xgboost, shap
from ucimlrepo import fetch_ucirepo
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from lime.lime_tabular import LimeTabularExplainer
from scipy.stats import spearmanr
from IPython.display import display
warnings.filterwarnings('ignore', category=FutureWarning)

MODE = 'FAST'  # İlk defa FAST; başarılı olunca PAPER yapın
assert MODE in ('FAST', 'PAPER')
N_PER_FOLD = 2 if MODE == 'FAST' else 8   # 5 dış kat -> 10/40 gözlem x 2 veri x 2 model x 2 olasılık
STABILITY_N_PER_GROUP = 1 if MODE == 'FAST' else 6
LIME_REPEATS = 3 if MODE == 'FAST' else 12
LIME_NUM_SAMPLES = 400 if MODE == 'FAST' else 1500
SHAP_BACKGROUND_N = 15 if MODE == 'FAST' else 35
SHAP_PERMUTATIONS = 2 if MODE == 'FAST' else 6
BOOTSTRAPS = 200 if MODE == 'FAST' else 2000
SEED = 42
MODELS = ['XGB_Unweighted','XGB_Weighted']
P_MODES = ['raw','calibrated']

zip_path = Path('/content/kredi_ek_deney_sonuclari_paper.zip')
if not zip_path.exists():
    from google.colab import files
    uploaded = files.upload()
    candidates = [Path(k) for k in uploaded if k.lower().endswith('.zip')]
    if len(candidates) != 1: raise ValueError('Lütfen yalnız kredi_ek_deney_sonuclari_paper.zip yükleyin.')
    zip_path=candidates[0]
OUT = Path('/content/kredi_xai_paper' if Path('/content').exists() else '/mnt/data/kredi_xai_paper')
OUT.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    manifest=json.loads(z.read('manifest.json'))
    tuning=pd.read_csv(io.BytesIO(z.read('tuning.csv')))
    previous=pd.read_csv(io.BytesIO(z.read('pooled_predictions.csv')))
assert manifest['mode']=='PAPER' and manifest['outer_folds']==5 and manifest['random_state']==42, 'Önceki PAPER manifesti farklı.'
assert set(MODELS).issubset(tuning.Model.unique())
print('Manifest doğrulandı. Mode:',MODE, '| sürümler:',manifest['versions'])

## 2. Aynı ham veri ve ön işleme

In [ ]:
# UCI 573 için doğrudan ZIP içindeki SouthGermanCredit.asc dosyası
SG_DE=['laufkont','laufzeit','moral','verw','hoehe','sparkont','beszeit','rate','famges','buerge','wohnzeit','verm','alter','weitkred','wohn','bishkred','beruf','pers','telef','gastarb','kredit']
SG_EN=['status','duration','credit_history','purpose','amount','savings','employment_duration','installment_rate','personal_status_sex','other_debtors','present_residence','property','age','other_installment_plans','housing','number_credits','job','people_liable','telephone','foreign_worker','credit_risk']

def load_data():
    t=fetch_ucirepo(id=350)
    Xt=t.data.features.copy().reset_index(drop=True).drop(columns=['ID'],errors='ignore')
    yt=t.data.targets.iloc[:,0].astype(int).reset_index(drop=True)
    Xt.columns=['LIMIT_BAL','SEX','EDUCATION','MARRIAGE','AGE','PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6','BILL_AMT1','BILL_AMT2','BILL_AMT3','BILL_AMT4','BILL_AMT5','BILL_AMT6','PAY_AMT1','PAY_AMT2','PAY_AMT3','PAY_AMT4','PAY_AMT5','PAY_AMT6']
    url='https://archive.ics.uci.edu/static/public/573/south+german+credit+update.zip'
    response=requests.get(url,timeout=90); response.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        candidates=[v for v in z.namelist() if Path(v).name.lower()=='southgermancredit.asc']
        if len(candidates)!=1: raise RuntimeError('UCI ZIP dosyasında beklenen veri bulunmadı.')
        raw=z.read(candidates[0])
    Xy=pd.read_csv(io.BytesIO(raw),sep=r'\s+',engine='python')
    if Xy.shape==(999,21) and all(str(k).isdigit() for k in Xy.columns):
        Xy=pd.read_csv(io.BytesIO(raw),sep=r'\s+',engine='python',header=None,names=SG_DE)
    if list(Xy.columns)==SG_DE: Xy.columns=SG_EN
    if list(Xy.columns)!=SG_EN: raise ValueError('South German sütun sırası beklenmiyor: '+str(Xy.columns.tolist()))
    Xy=Xy.apply(pd.to_numeric,errors='raise')
    yg=(Xy.pop('credit_risk').astype(int)==0).astype(int).reset_index(drop=True)
    Xg=Xy.reset_index(drop=True)
    assert Xt.shape==(30000,23) and int(yt.sum())==6636
    assert Xg.shape==(1000,20) and int(yg.sum())==300
    return {'Taiwan_Credit_Default':(Xt,yt,['SEX','EDUCATION','MARRIAGE']),
            'South_German_Credit':(Xg,yg,[c for c in Xg.columns if c not in ('duration','amount','age')])}

def make_prep(X,cats):
    num=[c for c in X.columns if c not in cats]
    return ColumnTransformer([
        ('num',Pipeline([('imp',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),
        ('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('ohe',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),cats)
    ],sparse_threshold=0.)

def get_xgb(X,cats,y,weighted,params):
    args=dict(objective='binary:logistic',eval_metric='logloss',tree_method='hist',n_jobs=1,random_state=42)
    if weighted: args['scale_pos_weight']=float((y==0).sum()/max(1,(y==1).sum()))
    model=Pipeline([('pre',make_prep(X,cats)),('clf',xgboost.XGBClassifier(**args))])
    return model.set_params(**json.loads(params))

def select_ids(y,ids,n,seed):
    rng=np.random.default_rng(seed)
    pos=[i for i in ids if int(y.iloc[i])==1]; neg=[i for i in ids if int(y.iloc[i])==0]
    take_pos=min(len(pos),n//2); take_neg=min(len(neg),n-take_pos)
    selected=list(rng.choice(pos,size=take_pos,replace=False))+list(rng.choice(neg,size=take_neg,replace=False))
    if len(selected)<n:
        rest=np.setdiff1d(ids,np.asarray(selected,dtype=int))
        selected+=list(rng.choice(rest,size=min(n-len(selected),len(rest)),replace=False))
    return sorted(map(int,selected))

datasets=load_data()
for ds,(X,y,cats) in datasets.items(): print(ds,'shape=',X.shape,'positive=',int(y.sum()))

## 3. Bir örnek için iki yöntemden açıklama alma

SHAP permutation masker, özellikleri **gerçek gözlemlerden** örneklenen referans değerlerle değiştirmektedir; bağımlı özelliklerde bu maskeleme doğal veri dağılımını korumaz. LIME ayrıca kategorik değişkenleri ayrık kategoriler olarak işleyip sürekli alanları bölümlere ayırır. Bu nedenle örtüşme, doğruluğun yerini tutan bir metrik değildir.

In [ ]:
def predict_fn(pipeline,columns):
    def fn(arr):
        arr=np.asarray(arr,dtype=float)
        if arr.ndim==1: arr=arr.reshape(1,-1)
        return pipeline.predict_proba(pd.DataFrame(arr,columns=columns))[:,1]
    return fn

def make_lime(Xtrain,features,cat_cols,seed):
    cats=[features.index(c) for c in cat_cols]
    return LimeTabularExplainer(Xtrain[features].to_numpy(dtype=float),mode='classification',
        feature_names=features,class_names=['NonDefault','Default'],categorical_features=cats,
        discretize_continuous=True,random_state=seed)

def local_lime(explainer,instance,positive_fn,features,num_samples):
    def predict_two(z):
        p=np.clip(positive_fn(z),0,1)
        return np.column_stack([1-p,p])
    exp=explainer.explain_instance(np.asarray(instance,dtype=float),predict_two,
                num_features=len(features),num_samples=num_samples)
    attributions=np.zeros(len(features),dtype=float)
    for j,v in exp.as_map()[1]: attributions[int(j)]=float(v)
    return attributions,float(exp.score)

def top_indices(x,k=5):
    return set(np.argsort(-np.abs(x),kind='stable')[:k].tolist())

def jaccard(a,b):
    u=set(a)|set(b)
    return float(len(set(a)&set(b))/len(u)) if u else 1.

def abs_spearman(a,b):
    if np.std(np.abs(a))<1e-12 or np.std(np.abs(b))<1e-12: return float('nan')
    return float(spearmanr(np.abs(a),np.abs(b)).statistic)

# Birkaç yardımcı fonksiyon kontrolü
assert jaccard({1,2,3},{2,3,4})==0.5
assert top_indices(np.array([0.8,0.1,0.5]),2)=={0,2}
print('XAI yardımcı fonksiyon kontrolleri geçti.')

## 4. Dış test örneklerinde açıklamalar (uzun süren hücre)

Her dış kat modeli **PAPER** sonuçlarının tuner parametreleriyle yeniden eğitilir. Önce ZIP'teki önceki tahminlerle eşleştiği doğrulanır; bu kontrol geçmezse makaleye bu sonuçları eklemeyin. Daha sonra SHAP–LIME uyumunu her koşulda aynı gözlemler üzerinde hesaplar.

In [ ]:
output_rows=[]; stability_rows=[]; reproducibility=[]; started=time.time()
for dataset_name,(X,y,cats) in datasets.items():
    outer_cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
    for fold,(itr,ite) in enumerate(outer_cv.split(X,y),start=1):
        Xtr=X.iloc[itr]; ytr=y.iloc[itr]; Xte=X.iloc[ite]
        selected=select_ids(y,ite,N_PER_FOLD,SEED+fold)
        cols=X.columns.tolist()
        bg=Xtr.sample(n=min(SHAP_BACKGROUND_N,len(Xtr)),random_state=SEED+fold)[cols].to_numpy(dtype=float)
        for model_name in MODELS:
            sel=tuning.query('Dataset==@dataset_name and Fold==@fold and Model==@model_name')
            assert len(sel)==1
            params=sel.iloc[0].Best_Params
            model=get_xgb(Xtr,cats,ytr,model_name.endswith('Weighted'),params)
            model.fit(Xtr,ytr)
            inner=StratifiedKFold(n_splits=3,shuffle=True,random_state=SEED+fold)
            calibrated=CalibratedClassifierCV(estimator=clone(model),method='sigmoid',cv=inner,ensemble=False)
            calibrated.fit(Xtr,ytr)
            for p_mode,pipe in [('raw',model),('calibrated',calibrated)]:
                prev=(previous[(previous.Dataset==dataset_name)&(previous.Fold==fold)&
                    (previous.Model==model_name)&(previous.Probability==p_mode)]
                    .set_index('RowID').sort_index())
                pred=pipe.predict_proba(Xte)[:,1]
                indexed_pred=pd.Series(pred,index=ite).loc[prev.index].to_numpy()
                max_err=float(np.max(np.abs(indexed_pred-prev.P.to_numpy())))
                reproducibility.append(dict(Dataset=dataset_name,Fold=fold,Model=model_name,
                                    Probability=p_mode,Maximum_Absolute_Difference=max_err))
                if max_err>1e-3:
                    raise RuntimeError(f'Önceki PAPER tahminleri yeniden üretilemedi ({dataset_name}/{fold}/{model_name}/{p_mode}, max_err={max_err:.6f})')
                fn=predict_fn(pipe,cols)
                masker=shap.maskers.Independent(bg,max_samples=len(bg))
                explainer=shap.PermutationExplainer(fn,masker,seed=SEED+fold)
                main_lime=make_lime(Xtr,cols,cats,seed=SEED+fold)
                for row_id in selected:
                    x=X.iloc[row_id][cols].to_numpy(dtype=float)
                    shap_values=explainer(x.reshape(1,-1),
                        max_evals=SHAP_PERMUTATIONS*2*len(cols)+1,
                        batch_size=200,silent=True)
                    sv=np.asarray(shap_values.values[0],dtype=float).reshape(-1)
                    lv,fit=local_lime(main_lime,x,fn,cols,LIME_NUM_SAMPLES)
                    top_sh=top_indices(sv);top_li=top_indices(lv)
                    sbase=float(np.asarray(shap_values.base_values).reshape(-1)[0]);px=float(fn(x)[0])
                    output_rows.append(dict(Dataset=dataset_name,Fold=fold,Model=model_name,
                        Probability=p_mode,RowID=row_id,Y=int(y.iloc[row_id]),Prediction=px,
                        Jaccard5=jaccard(top_sh,top_li),SpearmanAbs=abs_spearman(sv,lv),
                        LIME_Local_R2=fit,SHAP_Additivity_Residual=abs(sbase+float(sv.sum())-px),
                        SHAP_Top5=json.dumps([cols[i] for i in np.argsort(-np.abs(sv))[:5]]),
                        LIME_Top5=json.dumps([cols[i] for i in np.argsort(-np.abs(lv))[:5]])))
                    # LIME'nin rassal kararlılığı aynı hasta/gözlemde sınanır.
                    if selected.index(row_id)<STABILITY_N_PER_GROUP:
                        repeated=[]
                        for seed_i in range(LIME_REPEATS):
                            lr=make_lime(Xtr,cols,cats,seed=SEED+seed_i*101)
                            weights,_=local_lime(lr,x,fn,cols,LIME_NUM_SAMPLES)
                            repeated.append(weights)
                        js=[];rhos=[]
                        for a,b in itertools.combinations(repeated,2):
                            js.append(jaccard(top_indices(a),top_indices(b)))
                            rhos.append(abs_spearman(a,b))
                        stability_rows.append(dict(Dataset=dataset_name,Fold=fold,Model=model_name,
                            Probability=p_mode,RowID=row_id,LIME_Seed_Repetitions=LIME_REPEATS,
                            Pairwise_Jaccard5_Mean=float(np.nanmean(js)),
                            Pairwise_SpearmanAbs_Mean=float(np.nanmean(rhos))))
                print(f'{dataset_name} Fold={fold} {model_name} {p_mode}: {len(selected)} explanations OK, max Δpredict={max_err:.2g}',flush=True)
                for file,rows in [('xai_cases.csv',output_rows),('lime_stability.csv',stability_rows),
                                  ('prediction_reproducibility.csv',reproducibility)]:
                    pd.DataFrame(rows).to_csv(OUT/file,index=False)
print('Hücre tamamlandı. dakika=',round((time.time()-started)/60,2))

## 5. Gruplara göre özet, kalibrasyon öncesi/sonrası ve ağırlık karşılaştırması

In [ ]:
res=pd.read_csv(OUT/'xai_cases.csv')
stab=pd.read_csv(OUT/'lime_stability.csv')
repro=pd.read_csv(OUT/'prediction_reproducibility.csv')
assert repro.Maximum_Absolute_Difference.max()<=1e-3
assert res.groupby(['Dataset','Model','Probability']).size().min()==5*N_PER_FOLD
summary=(res.groupby(['Dataset','Model','Probability'],as_index=False)
         .agg(N=('RowID','size'),Jaccard5_Mean=('Jaccard5','mean'),Jaccard5_SD=('Jaccard5','std'),
              SpearmanAbs_Mean=('SpearmanAbs','mean'),LIME_Local_R2_Mean=('LIME_Local_R2','mean'),
              SHAP_Additivity_Error_Max=('SHAP_Additivity_Residual','max')))
st_summary=(stab.groupby(['Dataset','Model','Probability'],as_index=False)
            .agg(N=('RowID','size'),LIME_Jaccard5_Mean=('Pairwise_Jaccard5_Mean','mean'),
                 LIME_SpearmanAbs_Mean=('Pairwise_SpearmanAbs_Mean','mean')))
summary=summary.merge(st_summary,on=['Dataset','Model','Probability'],how='left')
summary.to_csv(OUT/'xai_summary.csv',index=False)

def paired_ci(tab,a='raw',b='calibrated',Nboot=1000):
    z=tab.pivot(index='RowID',columns='Probability',values='Jaccard5').dropna()
    d=(z[a]-z[b]).to_numpy();rng=np.random.default_rng(2026)
    bootstrap=np.array([d[rng.integers(0,len(d),len(d))].mean() for _ in range(Nboot)])
    return len(d),float(np.mean(d)),float(np.quantile(bootstrap,0.025)),float(np.quantile(bootstrap,0.975))
comparisons=[]
for (ds,model),sub in res.groupby(['Dataset','Model']):
    n,d,lo,hi=paired_ci(sub,Nboot=BOOTSTRAPS)
    comparisons.append(dict(Dataset=ds,Model=model,Comparison='Raw_minus_Calibrated_Jaccard5',N=n,
                            Difference=d,CI_Lower=lo,CI_Upper=hi))
pd.DataFrame(comparisons).to_csv(OUT/'xai_paired_comparisons.csv',index=False)
display(summary.round(4))
display(pd.DataFrame(comparisons).round(4))
print('Tüm tahminler yeniden üretildi: maksimum fark=',repro.Maximum_Absolute_Difference.max())

## 6. Dosyaları ZIP olarak gönderin

Bütün deneyler başarıyla tamamlandıysa yeni ZIP'i bu konuşmaya yükleyin. Nihai makalede eski SHAP–LIME tabloları yerine aynı tahmin uzayında üretilen bu deneyler kullanılacak. LIME kararlılığı ve SHAP–LIME uyumu birbirinden ayrı raporlanacak.

In [ ]:
meta=dict(mode=MODE,explain_per_fold=N_PER_FOLD,stability_instances_per_fold=STABILITY_N_PER_GROUP,
          lime_repeats=LIME_REPEATS,lime_num_samples=LIME_NUM_SAMPLES,
          shap_background=SHAP_BACKGROUND_N,shap_permutations=SHAP_PERMUTATIONS,
          n_cases=int(len(res)),n_stability=int(len(stab)),
          reproducibility_max_error=float(repro.Maximum_Absolute_Difference.max()),
          input_manifest=manifest)
(OUT/'xai_manifest.json').write_text(json.dumps(meta,indent=2,ensure_ascii=False),encoding='utf-8')
zip_output=Path('/content/kredi_xai_sonuclari_'+MODE.lower()+'.zip' if Path('/content').exists() else '/mnt/data/kredi_xai_sonuclari_'+MODE.lower()+'.zip')
with zipfile.ZipFile(zip_output,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for f in OUT.glob('*.csv'): z.write(f,arcname=f.name)
    z.write(OUT/'xai_manifest.json',arcname='xai_manifest.json')
print('Hazır:',zip_output)
from google.colab import files
files.download(str(zip_output))